In [0]:
from pyspark.sql.functions import *
from pyspark.sql import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("Storage_name", "sthmsdev001")
dbutils.widgets.text("container_name", "landingfiles")
Storage_name = dbutils.widgets.get("Storage_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
spark.conf.set(
    f"fs.azure.account.auth.type.{Storage_name}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{Storage_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_hms_scope", key="app-hms-clinet-id"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_hms_scope", key="app-hms-client-secret"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{Storage_name}.dfs.core.windows.net",
    "https://login.microsoftonline.com/65583477-d432-469c-b4af-ab046853916f/oauth2/v2.0/token",
)



In [0]:
source_path = f"abfss://{container_name}@{Storage_name}.dfs.core.windows.net/"
tables = ["bookings", "guests","hotels","payments","reviews","rooms"]

for table_name in tables:
    full_source_path = source_path + table_name + ".csv"
    
    #Read Data from landing files
    df = (spark.read.format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .load(full_source_path)\
        .withColumn("Action_TimeStamp",current_timestamp())\
        .withColumn("Created_by",lit("Ank1221"))\
        .withColumn("Batch_id",date_format(current_timestamp(),"yyyyMMddHHmmss"))\
        .withColumn("Source_file_name",input_file_name()))
    
    #Write data into Bronze layer
    df.write.format("delta").mode("overwrite").saveAsTable(f"hms_catalog.hms_Bronze.{table_name}")
    

    